# Web Analytics Performance Analysis

## Business Objective

Analyze website traffic and engagement data to understand **which acquisition channels attract traffic, which channels generate stronger engagement, and when website demand is highest**.

This project is intentionally focused on the metrics available in the dataset. It does **not** claim revenue, conversion, customer lifetime value, or ROI because those fields are not present.

### Business questions

1. How large is the website's session activity?
2. How many sessions are engaged?
3. What is the overall engagement rate?
4. Which channels generate the most sessions?
5. Which channels generate the strongest engagement rate?
6. Which channels keep users engaged for longer?
7. Which channels generate the highest event activity?
8. How does website traffic change over time?
9. Which hours have the highest session volume?
10. Which channel-hour combinations have the strongest traffic?
11. Which channels combine high traffic volume with strong engagement quality?
12. Are high-traffic channels also high-quality channels?
13. Where should acquisition and content teams prioritize optimization based on traffic and engagement evidence?

> **Important metric limitation:** the source is hourly/channel-level analytics data. `Users` should not be summed across hours and presented as unique users. This notebook therefore uses sessions, engaged sessions, engagement rate, engagement time, and event activity as the primary comparable KPIs. Where users are shown, they are explicitly labelled as reported user counts across records, not unique users.

## 1. Analytical Framework

```text
Raw GA-style Web Analytics Data
            ↓
Data Quality & Structure Checks
            ↓
Cleaning + Type Conversion
            ↓
Derived Time Dimensions
            ↓
Core KPIs
            ↓
Channel Performance
            ↓
Time & Hour Analysis
            ↓
Volume vs Engagement Quality
            ↓
Business Insights
            ↓
Recommendations
```

### Tools

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")

## 2. Load the Dataset

The original notebook used a machine-specific Windows path (`C:\Users\hp\Downloads\web_data.csv`). That makes the project difficult for another person to reproduce.

The improved notebook loads the **repository dataset** using a relative path so it works after cloning the project.

In [ ]:
DATA_PATH = Path("../data/web_data.csv")
CLEANED_PATH = Path("../data/cleaned_web_data.csv")

# Prefer the raw repository file when it contains data; otherwise use the cleaned file.
if DATA_PATH.exists() and DATA_PATH.stat().st_size > 0:
    file_path = DATA_PATH
elif CLEANED_PATH.exists():
    file_path = CLEANED_PATH
else:
    raise FileNotFoundError(
        "Dataset not found. Place web_data.csv or cleaned_web_data.csv in ../data/"
    )

df_raw = pd.read_csv(file_path, header=None)

print(f"Loaded: {file_path}")
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 3. Fix the Exported Header

The source file is a Google Analytics-style export where the first row contains the real column names. The original notebook initially loaded the file without treating that row as the header, which created columns such as `Unnamed: 1`.

The improved workflow explicitly promotes the first row to the header and removes the duplicate header row.

In [ ]:
# First row contains the actual field names
df = pd.read_csv(file_path, header=None)

df.columns = df.iloc[0].astype(str).str.strip()
df = df.iloc[1:].copy()

# Standardize column names
rename_map = {
    "Session primary channel group (Default channel group)": "Channel group",
    "Date + hour (YYYYMMDDHH)": "Datehour",
    "Users": "Users",
    "Sessions": "Sessions",
    "Engaged sessions": "Engaged sessions",
    "Average engagement time per session": "Avg engagement time",
    "Engaged sessions per user": "Engaged sessions per user",
    "Events per session": "Events per session",
    "Engagement rate": "Engagement rate",
    "Event count": "Event count",
}

df = df.rename(columns=rename_map)

expected_columns = [
    "Channel group", "Datehour", "Users", "Sessions", "Engaged sessions",
    "Avg engagement time", "Engaged sessions per user",
    "Events per session", "Engagement rate", "Event count"
]

missing_columns = [c for c in expected_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df = df[expected_columns].copy()

print(f"Cleaned shape: {df.shape}")
df.head()

## 4. Data Quality Checks

Before calculating KPIs, check:

- Dataset dimensions
- Data types
- Missing values
- Duplicate rows
- Invalid numeric values
- Channel categories
- Date coverage
- Metric ranges

These checks make the analysis reproducible and help prevent misleading conclusions.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nChannel groups:")
display(df["Channel group"].value_counts(dropna=False).to_frame("row_count"))

print("\nDatehour sample:")
display(df["Datehour"].head().to_frame())

In [ ]:
# Convert numeric columns
numeric_cols = [
    "Users", "Sessions", "Engaged sessions",
    "Avg engagement time", "Engaged sessions per user",
    "Events per session", "Engagement rate", "Event count"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Parse YYYYMMDDHH
df["Datehour"] = pd.to_datetime(
    df["Datehour"].astype(str).str.replace(".0", "", regex=False),
    format="%Y%m%d%H",
    errors="coerce"
)

# Remove rows that cannot be analysed
invalid_date_rows = df["Datehour"].isna().sum()
df = df.dropna(subset=["Channel group", "Datehour"]).copy()

print(f"Rows with invalid Datehour removed: {invalid_date_rows}")
print(f"Final analytical rows: {len(df):,}")

In [ ]:
# Range and logic checks
quality_checks = pd.DataFrame({
    "Check": [
        "Negative sessions",
        "Negative engaged sessions",
        "Engaged sessions > sessions",
        "Engagement rate outside 0-1",
        "Negative event count",
        "Negative engagement time"
    ],
    "Count": [
        (df["Sessions"] < 0).sum(),
        (df["Engaged sessions"] < 0).sum(),
        (df["Engaged sessions"] > df["Sessions"]).sum(),
        ((df["Engagement rate"] < 0) | (df["Engagement rate"] > 1)).sum(),
        (df["Event count"] < 0).sum(),
        (df["Avg engagement time"] < 0).sum()
    ]
})

display(quality_checks)

print("Date range:", df["Datehour"].min(), "to", df["Datehour"].max())
print("Channels:", df["Channel group"].nunique())

## 5. Feature Engineering

Create reusable time dimensions and a more defensible engagement-rate metric.

### Why recalculate engagement rate?

The source contains an engagement-rate field for each hourly/channel record. For an overall or channel-level KPI, simply averaging those percentages gives each row equal weight, even when rows have very different session volumes.

Instead:

**Weighted engagement rate = total engaged sessions / total sessions**

This gives high-volume records appropriate weight.

In [ ]:
df["Date"] = df["Datehour"].dt.date
df["Month"] = df["Datehour"].dt.to_period("M").astype(str)
df["Day"] = df["Datehour"].dt.day
df["Hour"] = df["Datehour"].dt.hour
df["Day of Week"] = df["Datehour"].dt.day_name()

df["Non-engaged sessions"] = (
    df["Sessions"] - df["Engaged sessions"]
).clip(lower=0)

df["Calculated engagement rate"] = np.where(
    df["Sessions"] > 0,
    df["Engaged sessions"] / df["Sessions"],
    np.nan
)

df["Events per session (calculated)"] = np.where(
    df["Sessions"] > 0,
    df["Event count"] / df["Sessions"],
    np.nan
)

df.head()

# 6. Executive KPI Overview

These are the primary KPIs supported by the available data.

> **Note:** `Users` is not presented as a total unique-user KPI because the dataset is split by hour and channel. Summing it across rows can double-count users.

In [ ]:
total_sessions = df["Sessions"].sum()
total_engaged_sessions = df["Engaged sessions"].sum()
total_events = df["Event count"].sum()

overall_engagement_rate = (
    total_engaged_sessions / total_sessions
    if total_sessions > 0 else np.nan
)

weighted_avg_engagement_time = np.average(
    df["Avg engagement time"],
    weights=df["Sessions"]
) if df["Sessions"].sum() > 0 else np.nan

events_per_session = (
    total_events / total_sessions
    if total_sessions > 0 else np.nan
)

reported_user_sum = df["Users"].sum()

kpis = pd.DataFrame({
    "KPI": [
        "Total Sessions",
        "Engaged Sessions",
        "Overall Engagement Rate",
        "Weighted Avg Engagement Time / Session (sec)",
        "Total Events",
        "Events per Session",
        "Reported User Count Across Records*"
    ],
    "Value": [
        total_sessions,
        total_engaged_sessions,
        overall_engagement_rate,
        weighted_avg_engagement_time,
        total_events,
        events_per_session,
        reported_user_sum
    ]
})

display(kpis)

print("* Reported users are summed across hourly/channel records and should not be interpreted as unique users.")

## 7. Channel Performance

### Business questions

- Which channels generate the most sessions?
- Which channels have the strongest engagement quality?
- Which channels keep sessions engaged for longer?
- Which channels generate the most event activity?

For channel-level engagement rate, use:

**Total engaged sessions ÷ total sessions**

rather than a simple mean of row-level percentages.

In [ ]:
channel = (
    df.groupby("Channel group")
      .agg(
          Sessions=("Sessions", "sum"),
          Engaged_Sessions=("Engaged sessions", "sum"),
          Events=("Event count", "sum"),
          Reported_Users=("Users", "sum")
      )
      .reset_index()
)

channel["Engagement Rate"] = (
    channel["Engaged_Sessions"] / channel["Sessions"]
)

channel["Events per Session"] = (
    channel["Events"] / channel["Sessions"]
)

channel["Session Share"] = (
    channel["Sessions"] / channel["Sessions"].sum()
)

# Session-weighted average engagement time
weighted_time = (
    df.groupby("Channel group")
      .apply(
          lambda g: np.average(
              g["Avg engagement time"],
              weights=g["Sessions"]
          ) if g["Sessions"].sum() > 0 else np.nan
      )
      .rename("Weighted Avg Engagement Time")
)

channel = channel.merge(weighted_time, on="Channel group")

channel = channel.sort_values("Sessions", ascending=False)

display(channel.style.format({
    "Sessions": "{:,.0f}",
    "Engaged_Sessions": "{:,.0f}",
    "Events": "{:,.0f}",
    "Reported_Users": "{:,.0f}",
    "Engagement Rate": "{:.2%}",
    "Events per Session": "{:.2f}",
    "Session Share": "{:.2%}",
    "Weighted Avg Engagement Time": "{:.2f}"
}))

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = channel.sort_values("Sessions", ascending=True)
plt.barh(plot_df["Channel group"], plot_df["Sessions"])
plt.title("Sessions by Acquisition Channel")
plt.xlabel("Sessions")
plt.ylabel("Channel")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = channel.sort_values("Engagement Rate", ascending=True)
plt.barh(plot_df["Channel group"], plot_df["Engagement Rate"] * 100)
plt.title("Engagement Rate by Acquisition Channel")
plt.xlabel("Engagement Rate (%)")
plt.ylabel("Channel")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = channel.sort_values("Weighted Avg Engagement Time", ascending=True)
plt.barh(plot_df["Channel group"], plot_df["Weighted Avg Engagement Time"])
plt.title("Session-Weighted Engagement Time by Channel")
plt.xlabel("Average Engagement Time per Session (seconds)")
plt.ylabel("Channel")
plt.tight_layout()
plt.show()

## 8. Traffic Trend Over Time

### Business question

**How does website session demand change over time?**

Use daily aggregation for a cleaner business view rather than plotting every hourly timestamp.

In [ ]:
daily = (
    df.groupby("Date")
      .agg(
          Sessions=("Sessions", "sum"),
          Engaged_Sessions=("Engaged sessions", "sum"),
          Events=("Event count", "sum")
      )
      .reset_index()
)

daily["Engagement Rate"] = (
    daily["Engaged_Sessions"] / daily["Sessions"]
)

daily.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(daily["Date"], daily["Sessions"], label="Sessions")
ax1.set_xlabel("Date")
ax1.set_ylabel("Sessions")
ax1.tick_params(axis="x", rotation=45)

plt.title("Daily Website Sessions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily["Date"], daily["Engagement Rate"] * 100)
plt.title("Daily Engagement Rate")
plt.xlabel("Date")
plt.ylabel("Engagement Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Hour-of-Day Analysis

### Business questions

- Which hours have the highest traffic?
- When is the website most active?
- Does traffic intensity vary by channel?

This can support scheduling, content publishing, campaign timing, and monitoring decisions.

In [ ]:
hourly = (
    df.groupby("Hour")
      .agg(
          Sessions=("Sessions", "sum"),
          Engaged_Sessions=("Engaged sessions", "sum"),
          Events=("Event count", "sum")
      )
      .reset_index()
)

hourly["Engagement Rate"] = (
    hourly["Engaged_Sessions"] / hourly["Sessions"]
)

display(hourly.sort_values("Sessions", ascending=False).head(10))

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(hourly["Hour"], hourly["Sessions"], marker="o")
plt.title("Sessions by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Sessions")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 10. Channel × Hour Traffic

### Business question

**Which acquisition channels are strongest at different hours of the day?**

This is more actionable than looking only at total channel volume.

In [ ]:
hour_channel = (
    df.groupby(["Hour", "Channel group"])["Sessions"]
      .sum()
      .unstack(fill_value=0)
)

plt.figure(figsize=(12, 6))
sns.heatmap(hour_channel, cmap="YlGnBu")
plt.title("Session Volume by Hour and Acquisition Channel")
plt.xlabel("Channel")
plt.ylabel("Hour of Day")
plt.tight_layout()
plt.show()

## 11. Engaged vs Non-Engaged Sessions

### Business question

**Where is the largest gap between traffic volume and engaged traffic?**

A channel can generate many sessions but still have weaker engagement quality.

In [ ]:
channel_engagement = channel[[
    "Channel group", "Sessions", "Engaged_Sessions"
]].copy()

channel_engagement["Non-engaged Sessions"] = (
    channel_engagement["Sessions"] -
    channel_engagement["Engaged_Sessions"]
)

melted = channel_engagement.melt(
    id_vars="Channel group",
    value_vars=["Engaged_Sessions", "Non-engaged Sessions"],
    var_name="Session Type",
    value_name="Sessions"
)

plt.figure(figsize=(11, 5))
sns.barplot(
    data=melted,
    x="Channel group",
    y="Sessions",
    hue="Session Type"
)
plt.title("Engaged vs Non-Engaged Sessions by Channel")
plt.xlabel("Channel")
plt.ylabel("Sessions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Volume vs Engagement Quality

### Business question

**Which channels combine meaningful traffic volume with strong engagement?**

This avoids judging a channel using only one metric.

- X-axis = sessions
- Y-axis = engagement rate
- Bubble size = event activity

Channels in the upper-right have both higher volume and stronger engagement quality.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    channel["Sessions"],
    channel["Engagement Rate"] * 100,
    s=np.maximum(channel["Events"] / channel["Events"].max() * 1200, 80),
    alpha=0.7
)

for _, row in channel.iterrows():
    plt.annotate(
        row["Channel group"],
        (row["Sessions"], row["Engagement Rate"] * 100),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Channel Volume vs Engagement Quality")
plt.xlabel("Sessions")
plt.ylabel("Engagement Rate (%)")
plt.tight_layout()
plt.show()

## 13. Event Activity

### Business question

**Which channels generate the highest event activity and events per session?**

Event volume can indicate interaction intensity, but it should not automatically be interpreted as conversions because the dataset does not identify business conversion events.

In [ ]:
event_rank = channel.sort_values("Events", ascending=False)

display(event_rank[[
    "Channel group",
    "Events",
    "Events per Session",
    "Sessions"
]])

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = channel.sort_values("Events", ascending=True)
plt.barh(plot_df["Channel group"], plot_df["Events"])
plt.title("Event Count by Acquisition Channel")
plt.xlabel("Event Count")
plt.ylabel("Channel")
plt.tight_layout()
plt.show()

## 14. Engagement Time vs Engagement Rate

### Business question

**Do channels with longer engagement time also show stronger engagement rates?**

This helps distinguish different types of engagement quality.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    channel["Weighted Avg Engagement Time"],
    channel["Engagement Rate"] * 100,
    s=100,
    alpha=0.75
)

for _, row in channel.iterrows():
    plt.annotate(
        row["Channel group"],
        (row["Weighted Avg Engagement Time"], row["Engagement Rate"] * 100),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Engagement Time vs Engagement Rate by Channel")
plt.xlabel("Weighted Avg Engagement Time (seconds)")
plt.ylabel("Engagement Rate (%)")
plt.tight_layout()
plt.show()

# 15. KPI Ranking Tables

These tables are useful for a recruiter because they make the analysis easy to audit.

In [ ]:
print("Top channels by session volume")
display(
    channel.nlargest(5, "Sessions")[[
        "Channel group", "Sessions", "Session Share", "Engagement Rate"
    ]]
)

print("Top channels by engagement rate")
display(
    channel.nlargest(5, "Engagement Rate")[[
        "Channel group", "Engagement Rate", "Sessions",
        "Weighted Avg Engagement Time"
    ]]
)

print("Top channels by engagement time")
display(
    channel.nlargest(5, "Weighted Avg Engagement Time")[[
        "Channel group", "Weighted Avg Engagement Time",
        "Engagement Rate", "Sessions"
    ]]
)

print("Top channels by event activity")
display(
    channel.nlargest(5, "Events")[[
        "Channel group", "Events", "Events per Session", "Sessions"
    ]]
)

# 16. Automated Insight Generator

Instead of hard-coding findings, generate the summary directly from the current dataset. This prevents the README or notebook from becoming inconsistent when the data changes.

In [ ]:
top_volume = channel.loc[channel["Sessions"].idxmax()]
top_engagement = channel.loc[channel["Engagement Rate"].idxmax()]
top_time = channel.loc[channel["Weighted Avg Engagement Time"].idxmax()]
top_events = channel.loc[channel["Events"].idxmax()]
peak_hour = hourly.loc[hourly["Sessions"].idxmax()]

print("Key findings")
print("-" * 60)
print(
    f"Highest session volume: {top_volume['Channel group']} "
    f"({top_volume['Sessions']:,.0f} sessions)."
)
print(
    f"Highest engagement rate: {top_engagement['Channel group']} "
    f"({top_engagement['Engagement Rate']:.2%})."
)
print(
    f"Highest weighted engagement time: {top_time['Channel group']} "
    f"({top_time['Weighted Avg Engagement Time']:.1f} seconds/session)."
)
print(
    f"Highest event activity: {top_events['Channel group']} "
    f"({top_events['Events']:,.0f} events)."
)
print(
    f"Peak traffic hour: {int(peak_hour['Hour']):02d}:00 "
    f"({peak_hour['Sessions']:,.0f} sessions)."
)

# 17. Business Insights

The analysis should be interpreted through four lenses:

### 1. Acquisition volume
Identify channels responsible for the largest share of sessions.

### 2. Engagement quality
Compare engagement rate and session-weighted engagement time rather than relying on traffic volume alone.

### 3. Interaction intensity
Use event count and events per session to understand how actively sessions interact with the website.

### 4. Timing
Use daily and hourly patterns to identify periods of high demand.

**Important:** the dataset does not contain conversion, revenue, campaign-cost, or customer-level attribution fields. Therefore, the project does not make claims about ROI, conversion performance, or revenue impact.

# 18. Business Recommendations

Recommendations should be tied to the actual rankings produced above.

### Recommendation 1 — Protect high-volume channels
Monitor the highest-volume channels closely because changes in their performance can materially affect overall session activity.

### Recommendation 2 — Investigate high-engagement channels
Channels with strong engagement rates and/or engagement time can be studied for content, audience, or landing-page characteristics that may be transferable to other channels.

### Recommendation 3 — Optimize low-quality high-volume traffic
A channel with substantial session volume but relatively weak engagement deserves investigation into landing pages, audience targeting, content relevance, and traffic quality.

### Recommendation 4 — Use hourly patterns for campaign timing
Use peak session hours as an input when planning publishing, campaign launches, monitoring coverage, or promotional activity.

### Recommendation 5 — Add conversion data in a future version
To move from engagement analytics to commercial performance analysis, connect this dataset with conversion events, leads, purchases, campaign spend, or revenue.

# 19. Limitations & Data Caveats

1. **Users are not treated as unique users across the full dataset.** Hourly/channel rows can contain overlapping users, so summing `Users` can overcount unique people.
2. The dataset contains traffic and engagement metrics but no revenue or conversion fields.
3. No campaign-cost data is available, so ROI cannot be calculated.
4. Channel-level comparisons are based on aggregated records.
5. Engagement rate is recalculated using engaged sessions / sessions for weighted aggregation.
6. Event count measures interactions but does not identify which events are commercially meaningful.
7. The analysis covers the date range present in the supplied dataset; it should not be generalized beyond that period.

# 20. Conclusion

This project evaluates website performance using a business-oriented analytics framework:

**Traffic Volume → Engagement Quality → Interaction Intensity → Time Patterns → Business Recommendations**

The analysis demonstrates practical skills in:

- Data cleaning
- Data-quality validation
- Feature engineering
- KPI design
- Grouped analysis
- Time-series analysis
- Channel performance analysis
- Weighted metric calculation
- Data visualization
- Business interpretation

The project deliberately avoids unsupported metrics such as revenue, conversion rate, and ROI because those measures are not available in the source data.

# 21. Next Steps

Potential extensions:

- Add conversion-event data
- Add campaign spend
- Calculate conversion rate by channel
- Calculate cost per acquisition
- Measure channel ROI
- Add landing-page performance
- Add device and geographic segmentation
- Build a Power BI dashboard from the cleaned analytical dataset